# Exploración de las viviendas de Barcelona

Usad este notebook para conocer los datos **antes** de escribir `SPECS.md`. Lo que encontréis va a la sección **Datos de entrada y problemas** de `SPECS.md`, y las ideas para tratarlo, a **Estrategias y experimentos**.

Aquí se mira y se piensa: nada de lo que hagáis en este notebook entra en el modelo. Las transformaciones que decidáis se escriben en `SPECS.md`, y el agente las implementa en `src/`, dentro del `Pipeline` (o en `limpiar_objetivo()`, si son sobre el precio), para que se apliquen igual en la validación cruzada, en la validación y en el test. Este notebook es el único sitio donde escribís código vosotros.

El significado de cada columna está en `DATOS.md`. Recordad: los barrios son reales, pero las viviendas y los precios son simulados.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd().parent))  # para importar src/ desde notebooks/

from src.datos import cargar_entrenamiento, cargar_test

entrenamiento = cargar_entrenamiento()
test = cargar_test()
print(f"Entrenamiento: {entrenamiento.shape[0]:,d} filas · Test: {test.shape[0]:,d} filas")

Entrenamiento: 6,237 filas · Test: 4,002 filas


## 1. Perfil de cada columna

In [ ]:
def perfil(df: pd.DataFrame) -> pd.DataFrame:
    """Tipo, nulos, ceros, valores distintos y un ejemplo de cada columna."""
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(1),
        "ceros": (df == 0).sum(),
        "distintos": df.nunique(),
        "ejemplo": df.iloc[0],
    })

perfil(entrenamiento)

**¿Pasa lo mismo en el test?** El modelo tendrá que tasar esas 4.002 viviendas, con sus nulos y sus rarezas.

In [ ]:
pd.DataFrame({
    "% nulos entrenamiento": (entrenamiento.isna().mean() * 100).round(1),
    "% nulos test": (test.isna().mean() * 100).round(1),
}).dropna(how="all").query("`% nulos entrenamiento` > 0 or `% nulos test` > 0")

## 2. La variable objetivo

In [ ]:
px.histogram(entrenamiento, x="precio", nbins=100, template="simple_white",
             title="Precio de la operación", labels={"precio": "Precio (€)"})

- ¿Qué forma tiene la distribución? ¿Cuáles son los precios más bajos y los más altos? ¿Alguno es imposible?
- Repetid el histograma con el logaritmo del precio. ¿Qué cambia? ¿Qué implicaría para un modelo lineal?
- `estado_operacion` dice si la operación se cerró. ¿Qué significa el precio de un anuncio **retirado**? ¿Es comparable con el de una venta?

In [ ]:
entrenamiento.groupby("estado_operacion").agg(
    filas=("precio", "size"),
    precio_mediano=("precio", "median"),
    euros_m2_mediano=("precio", lambda p: (p / entrenamiento.loc[p.index, "superficie_construida"].replace(0, np.nan)).median()),
)

## 3. Preguntas de calidad para empezar

- ¿Hay superficies imposibles (0, enormes)? ¿Es siempre la construida mayor que la útil?
- ¿Qué valores toma `planta`? ¿Cómo la convertiríais en un número? ¿Qué es "Entlo" o "Pral"?
- ¿Qué valores toman `ascensor`, `parking` y `tipo`? ¿Cuántas formas distintas hay de decir lo mismo? ¿Qué significa "Opcional" en `parking`?
- ¿Qué significa un 0 en `terraza_m2`? ¿Y un vacío?
- ¿Hay años de construcción imposibles?
- ¿En qué formatos está `fecha_venta`?
- ¿Cuándo falta el barrio? ¿Siempre fuera de Barcelona?
- ¿Hay `ref_catastral` repetidas? ¿Es la misma operación dos veces o la misma vivienda vendida dos veces?
- `dias_en_venta`: ¿lo sabe el banco en el momento de tasar?
- ¿Cuántas operaciones tiene cada barrio y cada municipio? ¿Hay zonas del test que no aparezcan en el entrenamiento?

**Ejemplo:** los diez valores más frecuentes de `planta`

In [ ]:
entrenamiento["planta"].value_counts(dropna=False).head(10)

**Ejemplo:** zonas del test que no están en el entrenamiento

In [ ]:
zona_ent = entrenamiento["barri"].fillna(entrenamiento["municipi"])
zona_test = test["barri"].fillna(test["municipi"])
nuevas = ~zona_test.isin(zona_ent)
print(f"{nuevas.sum()} viviendas del test están en {zona_test[nuevas].nunique()} zonas sin operaciones en el entrenamiento")

In [ ]:
# Vuestras comprobaciones aquí

## 4. Relación con el precio

Antes de probar una transformación, mirad cómo se relaciona cada variable con el precio. Una relación curva, valores imposibles o unos pocos puntos muy alejados son pistas de qué estrategia probar.

In [ ]:
px.scatter(entrenamiento, x="superficie_construida", y="precio", color="municipi", opacity=0.4,
           template="simple_white", title="Precio frente a superficie construida",
           labels={"superficie_construida": "Superficie construida (m²)", "precio": "Precio (€)"})

- ¿Qué le pasa a este gráfico con ejes logarítmicos (`log_x=True, log_y=True`)?
- Repetidlo coloreando por `tipo` o por `estado`.
- Cread una columna con los €/m² y comparadla entre barrios: ¿cuánto varía?

In [ ]:
entrenamiento.select_dtypes("number").corr(method="spearman")["precio"].sort_values()

- ¿Por qué usamos la correlación de Spearman y no la de Pearson? (Sesión 06.)
- Una correlación baja, ¿quiere decir que la variable no sirve para tasar?

## 5. La ubicación

In [ ]:
con_coordenadas = entrenamiento.dropna(subset=["latitud", "longitud"])
px.scatter_map(con_coordenadas, lat="latitud", lon="longitud", color=np.log10(con_coordenadas["precio"].clip(lower=1e4)),
               opacity=0.6, zoom=10, height=600, map_style="carto-positron",
               title="Precio según la ubicación", labels={"color": "log₁₀ precio"})

- ¿Dónde están las viviendas caras? ¿Ves algún punto fuera de sitio?
- La ubicación está en varias columnas: `municipi`, `districte`, `barri`, las coordenadas y `distancia_centro_km`. ¿Cuál usaríais en un modelo lineal y por qué?

## 6. Ideas para características nuevas

Pensad como un tasador: ¿qué miraría que no esté directamente en una columna? Algunas pistas: la altura real de la vivienda, si hay que subir escaleras, la época del edificio, el momento de la venta.

Escribid cada idea en `SPECS.md` §4 con su hipótesis **antes** de probarla.

In [ ]:
# Vuestras ideas aquí